# Part 11 · Notebook 08 — Evaluation, cost and safety for every LLM feature

**Sessions:** S8 (Evaluation, cost, safety & M7b release) · [Lesson plan](../../docs/lessons/PART_11_AI_NLP_COWORK.md) · graded labs in [`labs/part11/`](../../labs/part11/)

**You will:**
1. Price a response from its usage, with and without cache and batch.
2. Measure a cache hit rate, and what caching saves on a day of headlines.
3. Gate a prompt change on a golden set: no regressions allowed.
4. Make answers reproducible and rate-limit the calls.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
Nothing calls the API: model outputs are recorded responses built as real `anthropic` Message objects, and the data is synthetic with known answers, so every check is exact and free.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p11lib.py is in notebooks/part11/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p11lib as p

p.use_course_style()

## 1. What a response costs

Every response carries `usage`: uncached `input_tokens`, `cache_read_input_tokens`, `cache_creation_input_tokens` and `output_tokens`. Multiply each by its price per million tokens (`p.PRICES`: illustrative; check the current price list) and sum; a batch job costs half. The cache counters may be `None`: treat them as 0.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def cost_usd(usage, prices=p.PRICES, batch=False):
    c = ... / 1e6                                                    # ✍️ the four components, each × its price
    return c / 2 if batch else c

usages = [p.make_message("x", input_tokens=3000, output_tokens=300).usage,
          p.make_message("x", input_tokens=200, output_tokens=300, cache_write=2800).usage,
          p.make_message("x", input_tokens=200, output_tokens=300, cache_read=2800).usage]
cases = [(u,) for u in usages] + [(usages[2], p.PRICES, True)]
mine = [p.attempt(cost_usd, *c) for c in cases]
mine = p.check("cost_usd", mine, [p.cost_usd(*c) for c in cases])
pd.Series(mine, index=["3,000 input tokens, no cache", "first call: rubric written to the cache", "later calls: rubric read from the cache",
                       "later calls, in a batch"], name="USD per call").round(5).to_frame()

The first cached call costs a little more (writing the cache); every later call pays a tenth of the price for the cached rubric. The **cache hit rate** is the share of input tokens served from the cache: `Σ reads / Σ (uncached + reads + writes)`, 0 when there were no calls.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def cache_hit_rate(usages):
    read = sum(u.cache_read_input_tokens or 0 for u in usages)
    total = ...                                                      # ✍️
    return read / total if total else 0.0

day = [usages[1]] + [usages[2]] * 799                              # one cache write, then 799 reads
cases = [day, [usages[0]] * 800, []]
mine = [p.attempt(cache_hit_rate, c) for c in cases]
mine = p.check("cache_hit_rate", mine, [p.cache_hit_rate(c) for c in cases])
pd.DataFrame({"no cache": {"cache hit rate": mine[1], "cost (USD)": 800 * p.cost_usd(usages[0])},
              "cached rubric": {"cache hit rate": mine[0], "cost (USD)": sum(p.cost_usd(u) for u in day)},
              "cached rubric + batch": {"cache hit rate": mine[0], "cost (USD)": sum(p.cost_usd(u, batch=True) for u in day)}}).T.round(3)

Caching the rubric and batching cut the day's bill by more than three quarters, with identical answers. Track the hit rate in production: if it drops to zero, something in the prefix changed (a timestamp in the system prompt is the classic).

## 2. A regression gate

A new prompt or model must not break what worked. On the golden set, compare old and new answers: the share correct for each, the **regressions** (indices right before and wrong now), and `passed` only if there are no regressions **and** the score didn't fall.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def regression_report(golds, old, new, same):
    ok_old = [same(o, g) for o, g in zip(old, golds)]
    ok_new = [same(n, g) for n, g in zip(new, golds)]
    regressions = ...                                                # ✍️ indices correct before and wrong now
    so, sn = float(np.mean(ok_old)), float(np.mean(ok_new))
    return {"old_score": so, "new_score": sn, "regressions": regressions, "passed": ...}   # ✍️

golds = ["raised", "lowered", "maintained", "raised", "lowered", "raised"]
old = ["raised", "lowered", "raised", "raised", "lowered", "maintained"]
new = ["raised", "maintained", "maintained", "raised", "lowered", "raised"]
same = lambda a, b: a == b                                            # noqa: E731
cases = [(golds, old, new, same), (golds, old, golds, same), (golds, old, old, same)]
mine = [p.attempt(regression_report, *c) for c in cases]
mine = p.check("regression_report", mine, [p.regression_report(*c) for c in cases])
mine[0]

The new prompt scores higher overall, yet it breaks item 1, which the old one got right: the gate fails. A higher average is not enough when a specific case (a specific client, filing or instrument) silently got worse.

## 3. Reproducibility and rate limits

`p.ResponseCache` keys every answer by `sha256(model | prompt_version | input)`: the same question under a new prompt version is a new key, so an answer is never served from a different prompt. `p.RateLimiter` allows at most `max_calls` per window (the clock is injected, so the demo is deterministic):

In [ ]:
cache, calls = p.ResponseCache(), []
answer = lambda text: calls.append(text) or f"summary of {text!r}"          # noqa: E731
for version, text in (("v1", "ACME 10-K"), ("v1", "ACME 10-K"), ("v2", "ACME 10-K"), ("v1", "BOLT 10-K")):
    cache.get_or_call(p.MODEL, version, text, answer)
print(f"4 requests → {len(calls)} model calls: {calls}")

clock = [0.0]
limiter = p.RateLimiter(max_calls=3, per_seconds=60, now=lambda: clock[0])
log = []
for t in (0, 10, 20, 30, 59, 61, 75):
    clock[0] = t
    log.append((t, limiter.allow()))
print("calls at t (s) → allowed:", log)

## The M7b release checklist

An LLM feature ships (milestone M7b) with: recorded responses and golden sets in the test suite; schema validation, grounding and source screening on every extraction; cited, sanitized RAG answers with faithfulness and refusal scores; agents that can only propose, with audits and budgets; signed endpoints and validated workflow exports; a regression gate on every prompt or model change; cost and cache-hit dashboards; rate limits. Clinic W2 runs the whole chain end to end: news → score → alert → brief → agent proposal → risk check → human decision → audit, plus a timed kill-switch drill.

## Wrap-up

* Know the cost of every response; cache stable prefixes; batch what can wait.
* Gate every prompt or model change on a golden set; no regressions.
* Key responses by model and prompt version; rate-limit every session.
* Graded version: `labs/part11/week40_agents_ops` (`cost_usd`, `cache_hit_rate`, `ResponseCache`, `regression_report`, `RateLimiter`) and Clinic W2.